In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import time

url = "https://www.cve.org/CVERecord/SearchResults?query=2026"

options = Options()
# Don't use headless yet — we want to see Chrome
# options.add_argument("--headless")

driver = webdriver.Chrome(options=options)

driver.get(url)

time.sleep(5)

print(driver.title)

In [15]:

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

url = "https://www.flipkart.com/q/mobiles-under-30000"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/153.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9"
}

response = requests.get(url, headers=headers, timeout=15)

print("Status Code:", response.status_code)

soup = BeautifulSoup(response.text, "html.parser")

# Find product links/cards
products = soup.find_all("a")

data = []

for product in products:

    text = product.get_text(" ", strip=True)

    # We are interested in elements containing prices
    if "₹" not in text:
        continue

    # Get product URL
    link = product.get("href")

    if not link:
        continue

    # Avoid very large/non-product elements
    if len(text) > 1000:
        continue

    data.append({
        "Product": text,
        "URL": link
    })


# Remove duplicates
df = pd.DataFrame(data).drop_duplicates()

print(df.head(20))

# Save data
df.to_csv(
    "flipkart_mobiles_under_30000.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved successfully!")
print("Total records:", len(df))

Status Code: 403
Empty DataFrame
Columns: []
Index: []

Saved successfully!
Total records: 0


In [17]:
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# Read HTML
with open("flipkart.html", "r", encoding="utf-8") as file:
    html = file.read()

# Create BeautifulSoup object
soup = BeautifulSoup(html, "html.parser")

# Find product cards
products = soup.select("div[data-id]")

print("Products found:", len(products))

for product in products:

    # Product name
    name_tag = product.select_one(".RG5Slk")

    # Product link
    link_tag = product.select_one("a")

    if name_tag:
        name = name_tag.get_text(strip=True)
    else:
        name = ""

    if link_tag:
        url = urljoin(
            "https://www.flipkart.com",
            link_tag.get("href", "")
        )
    else:
        url = ""

    print("Name:", name)
    print("URL:", url)
    print("-" * 80)

Products found: 1
Name: OnePlus N6 5G (Midnight Green, 128 GB)
URL: https://www.flipkart.com/oneplus-n6-5g-midnight-green-128-gb/p/itm73a3c7e824695?pid=MOBHZVFJXH4NR5V9&lid=LSTMOBHZVFJXH4NR5V9ZABPAZ&marketplace=FLIPKART&q=mobile+under+30000&store=tyy%2F4io&srno=s_1_1&otracker=search&otracker1=search&fm=organic&iid=1f0a73fd-91c9-4656-abec-4e87a95764e6.MOBHZVFJXH4NR5V9.SEARCH&ppt=dynamic&ppn=Login%3ACategory_List&ssid=3p2ojfgdw00000001789712312265&qH=68b16980d792dbd7&ov_redirect=true
--------------------------------------------------------------------------------


In [18]:
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd
import re


# ---------------------------------------
# 1. Read the saved Flipkart HTML
# ---------------------------------------

with open("flipkart.html", "r", encoding="utf-8") as file:
    html = file.read()


# ---------------------------------------
# 2. Create BeautifulSoup object
# ---------------------------------------

soup = BeautifulSoup(html, "html.parser")


# ---------------------------------------
# 3. Find product cards
# ---------------------------------------

products = soup.select("div[data-id]")

print("Products found:", len(products))


# ---------------------------------------
# 4. Extract first 10 products
# ---------------------------------------

top_10 = []

for product in products:

    # Product name
    name_tag = product.select_one(".RG5Slk")

    if not name_tag:
        continue

    name = name_tag.get_text(" ", strip=True)


    # Product URL
    link_tag = product.select_one("a")

    if link_tag:
        url = urljoin(
            "https://www.flipkart.com",
            link_tag.get("href", "")
        )
    else:
        url = ""


    # Get all text from product card
    text = product.get_text(" ", strip=True)


    # Try to find price
    price_match = re.search(
        r"₹[\d,]+",
        text
    )

    price = price_match.group(0) if price_match else ""


    # Try to find rating
    rating_match = re.search(
        r"\b[0-5]\.\d\b",
        text
    )

    rating = rating_match.group(0) if rating_match else ""


    # Save product
    top_10.append({
        "Rank": len(top_10) + 1,
        "Name": name,
        "Price": price,
        "Rating": rating,
        "URL": url,
        "Specifications": text
    })


    # Stop after 10
    if len(top_10) == 10:
        break


# ---------------------------------------
# 5. Create DataFrame
# ---------------------------------------

df = pd.DataFrame(top_10)


# ---------------------------------------
# 6. Display results
# ---------------------------------------

print(df.to_string(index=False))


# ---------------------------------------
# 7. Save to CSV
# ---------------------------------------

df.to_csv(
    "top_10_flipkart_mobiles_under_30000.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nFile saved successfully!")

Products found: 1
 Rank                                   Name   Price Rating                                                                                                                                                                                                                                                                                                                                                                                                                                URL                                                                                                                                                                                                                                         Specifications
    1 OnePlus N6 5G (Midnight Green, 128 GB) ₹26,830    4.2 https://www.flipkart.com/oneplus-n6-5g-midnight-green-128-gb/p/itm73a3c7e824695?pid=MOBHZVFJXH4NR5V9&lid=LSTMOBHZVFJXH4NR5V9ZABPAZ&marketplace=FLIPKART&q=mobile+under+30000&store=tyy%2F4io&srno=s_

In [21]:
import pandas as pd

df = pd.read_csv("top_10_flipkart_mobiles_under_30000.csv")
print(df)

   Rank                                    Name    Price  Rating  \
0     1  OnePlus N6 5G (Midnight Green, 128 GB)  ₹26,830     4.2   

                                                 URL  \
0  https://www.flipkart.com/oneplus-n6-5g-midnigh...   

                                      Specifications  
0  Add to Compare OnePlus N6 5G (Midnight Green, ...  


In [22]:
import csv

with open("top_10_flipkart_mobiles_under_30000.csv", "r") as file:

    reader = csv.reader(file)

    for row in reader:
        print(row)

['ï»¿Rank', 'Name', 'Price', 'Rating', 'URL', 'Specifications']
['1', 'OnePlus N6 5G (Midnight Green, 128 GB)', 'â‚¹26,830', '4.2', 'https://www.flipkart.com/oneplus-n6-5g-midnight-green-128-gb/p/itm73a3c7e824695?pid=MOBHZVFJXH4NR5V9&lid=LSTMOBHZVFJXH4NR5V9ZABPAZ&marketplace=FLIPKART&q=mobile+under+30000&store=tyy%2F4io&srno=s_1_1&otracker=search&otracker1=search&fm=organic&iid=1f0a73fd-91c9-4656-abec-4e87a95764e6.MOBHZVFJXH4NR5V9.SEARCH&ppt=dynamic&ppn=Login%3ACategory_List&ssid=3p2ojfgdw00000001789712312265&qH=68b16980d792dbd7&ov_redirect=true', 'Add to Compare OnePlus N6 5G (Midnight Green, 128 GB) 4.2 652 Ratings & 67 Reviews 6 GB RAM | 128 GB ROM 17.27 cm (6.8 inch) Display 50MP Rear Camera 8000 mAh Battery 1 Year Warranty â‚¹26,830 â‚¹33,999 21% off Upto â‚¹21,250 Off on Exchange Bank Offer']


In [23]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd


url = "https://www.flipkart.com/search?q=mobile%20under%2030000&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=on&as=off"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)

print("Status Code:", response.status_code)

soup = BeautifulSoup(response.text, "html.parser")


# Find all product cards
products = soup.select("div[data-id]")

print("Products found:", len(products))


data = []

for product in products[:10]:

    # -------------------------
    # Product Name
    # -------------------------

    name_tag = product.select_one(".RG5Slk")

    name = name_tag.get_text(strip=True) if name_tag else ""


    # -------------------------
    # Price
    # -------------------------

    price_tag = product.select_one(".hZ3P6w")

    price = price_tag.get_text(strip=True) if price_tag else ""


    # -------------------------
    # Rating
    # -------------------------

    rating_tag = product.select_one(".MKiFS6")

    rating = rating_tag.get_text(strip=True) if rating_tag else ""


    # -------------------------
    # Ratings & Reviews
    # -------------------------

    review_tag = product.select_one(".PvbNMB")

    reviews = review_tag.get_text(" ", strip=True) if review_tag else ""


    # -------------------------
    # Specifications
    # -------------------------

    specs = []

    for li in product.select(".HwRTzP li"):
        specs.append(li.get_text(" ", strip=True))


    # -------------------------
    # Product URL
    # -------------------------

    link_tag = product.select_one("a[href]")

    if link_tag:
        product_url = urljoin(
            "https://www.flipkart.com",
            link_tag["href"]
        )
    else:
        product_url = ""


    # -------------------------
    # Save product
    # -------------------------

    data.append({
        "Name": name,
        "Price": price,
        "Rating": rating,
        "Ratings & Reviews": reviews,
        "Specifications": " | ".join(specs),
        "URL": product_url
    })


# Create DataFrame
df = pd.DataFrame(data)

print(df)


# Save CSV
df.to_csv(
    "flipkart_top_10_mobiles.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nCSV saved successfully!")

Status Code: 403
Products found: 0
Empty DataFrame
Columns: []
Index: []

CSV saved successfully!


In [36]:
import requests

url = "https://science.nasa.gov/blogs/the-sun-spot/"
res = requests.get(url)
print(res.text)

<!doctype html>
<html lang="en-US" prefix="og: https://ogp.me/ns#">
<head>
	<meta charset="UTF-8">
<script>
var gform;gform||(document.addEventListener("gform_main_scripts_loaded",function(){gform.scriptsLoaded=!0}),document.addEventListener("gform/theme/scripts_loaded",function(){gform.themeScriptsLoaded=!0}),window.addEventListener("DOMContentLoaded",function(){gform.domLoaded=!0}),gform={domLoaded:!1,scriptsLoaded:!1,themeScriptsLoaded:!1,isFormEditor:()=>"function"==typeof InitializeEditor,callIfLoaded:function(o){return!(!gform.domLoaded||!gform.scriptsLoaded||!gform.themeScriptsLoaded&&!gform.isFormEditor()||(gform.isFormEditor()&&console.warn("The use of gform.initializeOnLoaded() is deprecated in the form editor context and will be removed in Gravity Forms 3.1."),o(),0))},initializeOnLoaded:function(o){gform.callIfLoaded(o)||(document.addEventListener("gform_main_scripts_loaded",()=>{gform.scriptsLoaded=!0,gform.callIfLoaded(o)}),document.addEventListener("gform/theme/scripts_l